# V2-3 (r3)  Comparison and report - CPU runtime is enough
Set `SEEDS` in the last cell to the seeds you ran (missing seed files are skipped).

In [ ]:
!pip install -q rasterio scikit-learn tqdm seaborn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
LIGHT = True
FEATURE_MODE = 'anomaly'
RUN_TAG = ''
DYN_USE = None

In [ ]:
# =====================================================================
# CORE v2 - shared setup (identical first code cell in every v2 notebook)
#   * one-step design: state(t) + drivers(t) -> state(t+10);   T1 = 2003->2013,  T2 = 2013->2023
#   * FEATURE_MODE 'anomaly' : each continuous layer standardised WITHIN its own year (removes year/sensor/weather offsets)
#     FEATURE_MODE 'global'  : old global min-max across years (kept only for the ablation)
# =====================================================================
import os, math, random, warnings, copy
import numpy as np, pandas as pd, cv2, rasterio
import matplotlib.pyplot as plt, seaborn as sns
from rasterio.warp import reproject, Resampling
from sklearn.metrics import confusion_matrix, cohen_kappa_score, precision_recall_fscore_support
warnings.filterwarnings('ignore')
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception:
    pass

LIGHT        = globals().get('LIGHT', False)               # True = skip feature building (used by the comparison notebook)
FEATURE_MODE = globals().get('FEATURE_MODE', 'anomaly')    # 'anomaly' | 'global'

# ---------------- CONFIG ----------------
PATH   = '/content/drive/MyDrive/Obj2_LULC_GWP/'
RUN_TAG = globals().get('RUN_TAG', '')                     # e.g. '_noclim' / '_stateonly' for ablation runs (separate output folder)
OUT    = os.path.join(PATH, f'revised_outputs_v2_{FEATURE_MODE}{RUN_TAG}'); os.makedirs(OUT, exist_ok=True)
YEARS  = ['2003', '2013', '2023']
CLASS_VALUES = [1, 2, 3, 4, 5]
CLASS_NAMES  = ['Water', 'Forest', 'Agriculture', 'Settlement', 'Fallow']
K = len(CLASS_VALUES); WATER_IDX, FOREST_IDX, FALLOW_IDX = 0, 1, 4
DYN_CONT = ['NDVI', 'MNDWI', 'LST', 'Rainfall', 'Dist_Water', 'NDMI']
_du = globals().get('DYN_USE', None)                       # None = all continuous layers; [] = state + static only
DYN_USE  = DYN_CONT if _du is None else _du
STATIC   = ['SRTM_DEM', 'Slope', 'DD', 'Ld', 'TWI']

PATCH, HIDDEN, EPOCHS, BATCH, LR, PATIENCE = 5, 64, 50, 1024, 5e-4, 6
SAMPLE_FRAC = 0.10
SEEDS = globals().get('SEEDS', [0, 1, 2])
BLOCK, SPLIT_SEED, TEST_FRAC, VAL_FRAC = 64, 42, 0.15, 0.15
CA_GAMMA = 1.0
STEEP_DEG = 10.0          # a-priori physical prior (NOT data-derived): water on slopes above this is implausible
PHYS_RAMP_DEG = 10.0      # penalty ramps 0 -> 1 between STEEP_DEG and STEEP_DEG + PHYS_RAMP_DEG

def set_seed(s):
    random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except ImportError: pass

# ---------------- RASTERS (reprojected onto the LULC_2023 master grid) ----------------
with rasterio.open(os.path.join(PATH, 'LULC_2023.tif')) as src:
    master_shape, master_meta, master_tf, master_crs = src.read(1).shape, src.meta.copy(), src.transform, src.crs
    PIXEL_KM2 = abs(src.res[0] * src.res[1]) / 1e6
H, W = master_shape
print(f'FEATURE_MODE = {FEATURE_MODE} | grid {H} x {W} | pixel = {PIXEL_KM2*1e6:.0f} m2 | outputs -> {OUT}')

def read_raster(name, categorical=False):
    with rasterio.open(os.path.join(PATH, name)) as src:
        a = src.read(1).astype('float32'); nd = src.nodata
        if nd is not None: a[a == nd] = np.nan
        a[(a < -1e10) | (a > 1e10)] = np.nan
        if src.transform != master_tf or a.shape != master_shape or src.crs != master_crs:
            dst = np.full(master_shape, np.nan, 'float32')
            reproject(source=a, destination=dst, src_transform=src.transform, src_crs=src.crs, src_nodata=np.nan,
                      dst_transform=master_tf, dst_crs=master_crs, dst_nodata=np.nan,
                      resampling=Resampling.nearest if categorical else Resampling.bilinear)
            print(f'  {name}: reprojected {a.shape} -> {master_shape}'); a = dst
    a[~np.isfinite(a)] = np.nan
    return a

def lulc_to_index(a):
    out = np.full(a.shape, -1, np.int8)
    for i, v in enumerate(CLASS_VALUES): out[a == v] = i
    return out

LULC  = {y: lulc_to_index(read_raster(f'LULC_{y}.tif', True)) for y in YEARS}
VALID = np.all([LULC[y] >= 0 for y in YEARS], axis=0)
VR, VC = np.where(VALID); NV = len(VR)
print(f'Valid pixels: {NV:,} ({NV*PIXEL_KM2:.1f} km2)')
slope_raw = read_raster('Slope.tif')
slope_deg = slope_raw.copy()
if np.nanmax(slope_deg) > 90: slope_deg = np.degrees(np.arctan(slope_deg / 100.0)); print('Slope treated as percent-rise -> degrees')
slope_deg = np.nan_to_num(slope_deg, nan=0.0)

# ---------------- FEATURES ----------------
def onehot(idx):
    oh = np.zeros((K,) + idx.shape, np.float32)
    for k in range(K): oh[k] = (idx == k)
    return oh

if not LIGHT:
    dyn_raw = {y: np.stack([read_raster(f'{n}_{y}.tif') for n in DYN_CONT]) for y in YEARS}
    stat_raw = np.stack([slope_raw if n == 'Slope' else read_raster(f'{n}.tif') for n in STATIC])
    for _y in YEARS:
        for _i, _n in enumerate(DYN_CONT):
            _p = np.isnan(dyn_raw[_y][_i][VALID]).mean() * 100
            if _p > 0.5: print(f'  CHECK {_n}_{_y}: {_p:.1f}% NaN inside study area')
    for _i, _n in enumerate(STATIC):
        _p = np.isnan(stat_raw[_i][VALID]).mean() * 100
        if _p > 0.5: print(f'  CHECK {_n}: {_p:.1f}% NaN inside study area')

    g_lo = np.array([np.nanmin([np.nanmin(dyn_raw[y][i][VALID]) for y in YEARS]) for i in range(len(DYN_CONT))])
    g_hi = np.array([np.nanmax([np.nanmax(dyn_raw[y][i][VALID]) for y in YEARS]) for i in range(len(DYN_CONT))])

    def feat_layer(a, i, name):
        if FEATURE_MODE == 'global':
            z = np.clip((a - g_lo[i]) / (g_hi[i] - g_lo[i] + 1e-6), 0, 1)
            z = np.where(np.isnan(z), np.nanmedian(z[VALID]), z)
        else:                                            # within-year standardisation (spatial anomaly), clipped to +-4 SD, scaled to [-1, 1]
            x = np.log1p(np.clip(a, 0, None)) if name == 'Dist_Water' else a
            v = x[VALID]; v = v[np.isfinite(v)]
            z = np.clip((x - v.mean()) / (v.std() + 1e-9), -4, 4) / 4
            z = np.where(np.isfinite(z), z, 0)
        return np.where(VALID, z, 0).astype(np.float32)

    def build_dyn(y):
        ch = [feat_layer(dyn_raw[y][i], i, n) for i, n in enumerate(DYN_CONT) if n in DYN_USE]
        return np.concatenate([onehot(LULC[y])] + ([np.stack(ch)] if ch else []))
    DYN = {y: build_dyn(y) for y in YEARS}                                     # (5 + n_cont, H, W) per year

    s_lo = np.nanmin(np.where(VALID, stat_raw, np.nan), axis=(1, 2))[:, None, None]
    s_hi = np.nanmax(np.where(VALID, stat_raw, np.nan), axis=(1, 2))[:, None, None]
    STAT = np.clip((stat_raw - s_lo) / (s_hi - s_lo + 1e-6), 0, 1)
    STAT = np.where(np.isnan(STAT), np.nanmedian(np.where(VALID, STAT, np.nan), axis=(1, 2), keepdims=True), STAT)
    STAT = np.where(VALID, STAT, 0).astype(np.float32)
    C_DYN, C_STAT = DYN['2003'].shape[0], STAT.shape[0]
    print(f'Channels: dynamic {C_DYN} (5 one-hot LULC + {C_DYN-K} continuous), static {C_STAT}')

    # year-shift check on pixels whose class never changed (should be small after standardisation)
    _st = (LULC['2003'] == LULC['2013']) & (LULC['2013'] == LULC['2023']) & VALID
    _names = [n for n in DYN_CONT if n in DYN_USE]; _mx = {}
    for j, n in enumerate(_names):
        _d = []
        for k in range(K):
            m = _st & (LULC['2023'] == k)
            if m.sum() > 500:
                _d.append([abs(DYN['2013'][K+j][m].mean() - DYN['2003'][K+j][m].mean()), abs(DYN['2023'][K+j][m].mean() - DYN['2013'][K+j][m].mean())])
        _mx[n] = np.max(_d, axis=0) * (4 if FEATURE_MODE == 'anomaly' else 1)
    print('Max class-conditional year shift on never-changed pixels (T1, T2) [SD units if anomaly mode, 0-1 units if global]:')
    for n, v in _mx.items(): print(f'   {n:11s} T1 {v[0]:.2f}   T2 {v[1]:.2f}')

# ---------------- SPATIAL BLOCK SPLIT ----------------
def make_block_split(valid, block, test_frac, val_frac, seed):
    nbr, nbc = math.ceil(H / block), math.ceil(W / block)
    has = np.array([[valid[i*block:(i+1)*block, j*block:(j+1)*block].any() for j in range(nbc)] for i in range(nbr)])
    ids = np.argwhere(has); rng = np.random.RandomState(seed); rng.shuffle(ids)
    n_te, n_va = int(round(test_frac * len(ids))), int(round(val_frac * len(ids)))
    lab = np.full((nbr, nbc), -1, np.int8)
    for n, (i, j) in enumerate(ids): lab[i, j] = 2 if n < n_te else (1 if n < n_te + n_va else 0)
    full = np.where(valid, np.repeat(np.repeat(lab, block, 0), block, 1)[:H, :W], -1).astype(np.int8)
    return full, np.repeat(np.repeat(np.arange(nbr*nbc).reshape(nbr, nbc), block, 0), block, 1)[:H, :W]
SPLIT, BLOCK_ID = make_block_split(VALID, BLOCK, TEST_FRAC, VAL_FRAC, SPLIT_SEED)
for n, nm in enumerate(['train', 'val', 'test']): print(f'  {nm:5s}: {(SPLIT==n).sum():,} px')

# ---------------- PATCH SOURCE (single epoch) ----------------
class PatchSource:
    def __init__(self, dyn, ps=PATCH):
        pad = ps // 2
        self.d = np.pad(dyn, ((0, 0), (pad, pad), (pad, pad)), mode='reflect'); self.off = np.arange(ps)
    def get(self, rows, cols):
        rr = (rows[:, None] + self.off[None, :])[:, :, None]; cc = (cols[:, None] + self.off[None, :])[:, None, :]
        x = self.d[:, rr, cc]                                           # (C,B,ps,ps)
        return (np.ascontiguousarray(np.transpose(x, (1, 0, 2, 3))),
                np.ascontiguousarray(STAT[:, rows, cols].T), slope_deg[rows, cols].astype(np.float32))

def sample_coords(split_id, frac, seed):
    r, c = np.where(SPLIT == split_id)
    sel = np.random.RandomState(seed).choice(len(r), int(len(r) * frac), replace=False)
    return r[sel], c[sel]

# ---------------- MARKOV DEMAND + CA ALLOCATION ----------------
def markov_matrix(a, b, mask):
    m = np.bincount((a[mask].astype(int) * K + b[mask]).astype(int), minlength=K*K).reshape(K, K).astype(float)
    return m / m.sum(1, keepdims=True).clip(1)

def allocate(T, demand, iters=300, tol=0.001):
    logT = np.log(T + 1e-9); b = np.zeros(K); N = T.shape[1]; lr = 1.0
    for _ in range(iters):
        lab = np.argmax(logT + b[:, None], 0); cnt = np.bincount(lab, minlength=K)
        dv = np.abs(cnt - demand).max() / N
        if dv < tol: break
        b += lr * np.log((demand + 1) / (cnt + 1)); lr = max(0.05, lr * 0.99)
    return lab.astype(np.int8), dv

def ca_allocate(prob, prev_map, trans, gamma=CA_GAMMA):
    """prob: (K, NV) class probabilities at valid pixels. Markov demand = class counts(prev) @ trans; 5x5 contiguity factor; competitive allocation."""
    S = prob.astype(np.float32).copy()
    for k in range(K): S[k] *= (1.0 + gamma * cv2.blur((prev_map == k).astype(np.float32), (5, 5))[VR, VC])
    demand = np.rint(np.bincount(prev_map[VALID], minlength=K) @ trans).astype(int)
    demand[np.argmax(demand)] += NV - demand.sum()
    lab, dv = allocate(S, demand)
    print(f'    demand km2 {(demand*PIXEL_KM2).round(1).tolist()} | max alloc-demand gap {dv*100:.3f}% px')
    return lab

def scatter(lab):
    out = np.full((H, W), -1, np.int8); out[VR, VC] = lab; return out

# ---------------- METRICS / IO ----------------
def evaluate_map(true, pred, prev, mask):
    t, p, q = true[mask].astype(np.int64), pred[mask].astype(np.int64), prev[mask].astype(np.int64)
    out = {'n': int(mask.sum()), 'OA': float((t == p).mean()), 'Kappa': float(cohen_kappa_score(t, p))}
    pr, rc, f1, _ = precision_recall_fscore_support(t, p, labels=range(K), zero_division=0)
    for k, nm in enumerate(CLASS_NAMES): out[f'F1_{nm}'] = float(f1[k]); out[f'R_{nm}'] = float(rc[k])
    out['MacroF1'] = float(f1.mean())
    ch = t != q
    hits, miss = int((ch & (p == t)).sum()), int((ch & (p == q)).sum())
    wrong, fa = int((ch & (p != q) & (p != t)).sum()), int((~ch & (p != q)).sum())
    out.update(change_observed=float(ch.mean()), change_predicted=float((p != q).mean()), hits=hits, misses=miss,
               wrong_hits=wrong, false_alarms=fa, FoM=hits / max(1, hits + miss + wrong + fa))
    qd = np.abs(np.bincount(t, minlength=K) - np.bincount(p, minlength=K)).sum() / 2 / len(t)
    out.update(QuantityDisagreement=float(qd), AllocationDisagreement=float((1 - out['OA']) - qd))
    steep = slope_deg[mask] > STEEP_DEG
    out.update(water_steep_pred=int(((p == WATER_IDX) & steep).sum()), water_steep_true=int(((t == WATER_IDX) & steep).sum()),
               new_water_pred=int(((p == WATER_IDX) & (q != WATER_IDX)).sum()), new_water_true=int(((t == WATER_IDX) & (q != WATER_IDX)).sum()),
               new_water_steep_pred=int(((p == WATER_IDX) & (q != WATER_IDX) & steep).sum()),
               F2F_pred_px=int(((q == FALLOW_IDX) & (p == FOREST_IDX)).sum()), F2F_true_px=int(((q == FALLOW_IDX) & (t == FOREST_IDX)).sum()),
               F2F_hit_px=int(((q == FALLOW_IDX) & (p == FOREST_IDX) & (t == FOREST_IDX)).sum()))
    return out

def save_map(idx_map, fname):
    m = master_meta.copy(); m.update(dtype='uint8', count=1, nodata=0, compress='lzw')
    codes = np.zeros(idx_map.shape, np.uint8)
    for k, v in enumerate(CLASS_VALUES): codes[(idx_map == k) & VALID] = v
    with rasterio.open(os.path.join(OUT, fname), 'w', **m) as dst: dst.write(codes, 1)

def read_idx(folder, fname):
    with rasterio.open(os.path.join(folder, fname)) as s: a = s.read(1)
    o = np.full(a.shape, -1, np.int8)
    for k, v in enumerate(CLASS_VALUES): o[a == v] = k
    return o
print('Core v2 ready.')


In [ ]:
# =====================================================================
# COMPARISON & REPORT (v2) - run after the CA and CNN notebooks (for one or both FEATURE_MODEs)
# =====================================================================
SEEDS = [0, 1, 2, 3, 4]      # <-- seeds you ran; missing files are skipped
MODES = ['anomaly', 'global', 'anomaly_noclim', 'anomaly_stateonly']   # whichever have outputs are used
BOOT_REGION = 'all_valid'              # bootstrap over ALL blocks (out-of-time test makes this legitimate and ~7x more powerful); or 'test_blocks'
TAGS  = {'CA_ANN_Markov': ['T2', 'T1', 'Mean'], 'CNN_argmax': ['na'], 'CNN_Markov': ['T2', 'T1', 'Mean'],
         'PI_CNN_argmax': ['na'], 'PI_CNN_Markov': ['T2', 'T1', 'Mean']}
PRIMARY_SCEN, REPORT_MODE = 'T2', 'anomaly'
def folder(m): return os.path.join(PATH, f'revised_outputs_v2_{m}')
def ok(m, f): return os.path.exists(os.path.join(folder(m), f))
REP = os.path.join(PATH, 'revised_outputs_v2_REPORT'); os.makedirs(REP, exist_ok=True)
TEST, ALLV = SPLIT == 2, VALID
STABLE_T1 = (LULC['2003'] == LULC['2013']) & VALID
STABLE_ALL = STABLE_T1 & (LULC['2013'] == LULC['2023'])
def shield(m, kind): return np.where(STABLE_T1 if kind == 'hind' else STABLE_ALL, LULC['2013'] if kind == 'hind' else LULC['2023'], m)
def vote(maps):
    st = np.stack(maps); return np.where(VALID, np.stack([(st == k).sum(0) for k in range(K)]).argmax(0), -1).astype(np.int8)

# ---------- 1. HINDCAST (out-of-time) metrics ----------
rows = []
for reg, mask in [('test_blocks', TEST), ('all_valid', ALLV)]:
    r = evaluate_map(LULC['2023'], LULC['2013'], LULC['2013'], mask); r.update(mode='-', model='Persistence', variant='-', seed=-1, region=reg); rows.append(r)
HIND = {}                                                   # (mode, tag) -> list of maps
for mode in MODES:
    for tag in TAGS:
        maps = [read_idx(folder(mode), f'{tag}_hind2023_seed{s}.tif') for s in SEEDS if ok(mode, f'{tag}_hind2023_seed{s}.tif')]
        if not maps: continue
        HIND[(mode, tag)] = maps
        for s, m in zip(SEEDS, maps):
            for var, mm in [('noshield', m), ('shield', shield(m, 'hind'))]:
                for reg, mask in [('test_blocks', TEST), ('all_valid', ALLV)]:
                    r = evaluate_map(LULC['2023'], mm, LULC['2013'], mask); r.update(mode=mode, model=tag, variant=var, seed=s, region=reg); rows.append(r)
res = pd.DataFrame(rows); res.to_csv(os.path.join(REP, 'hindcast_metrics_all.csv'), index=False)
cols = ['OA', 'Kappa', 'MacroF1', 'FoM', 'QuantityDisagreement', 'AllocationDisagreement', 'F1_Water', 'F1_Forest', 'F1_Fallow', 'change_predicted',
        'new_water_steep_pred', 'F2F_pred_px', 'F2F_hit_px']
g = res[res.region == 'test_blocks'].groupby(['mode', 'model', 'variant'])[cols]
summ = g.mean().round(4).astype(str) + ' ± ' + g.std().fillna(0).round(4).astype(str)
pers = res[(res.model == 'Persistence') & (res.region == 'test_blocks')].iloc[0]
print(f'OUT-OF-TIME VALIDATION (train 2003->2013, predict 2023 from 2013), test blocks. Observed change {pers.change_observed*100:.2f}%; '
      f'persistence OA {pers.OA*100:.2f}%; observed Fallow->Forest px {int(pers.F2F_true_px)}; observed new-water px {int(pers.new_water_true)}')
display(summ); summ.to_csv(os.path.join(REP, 'hindcast_summary_mean_sd.csv'))

# ---------- 2. Block-bootstrap contrasts (ensemble maps, test blocks) ----------
BM = ALLV if BOOT_REGION == 'all_valid' else TEST
ids = BLOCK_ID[BM]; ub, inv = np.unique(ids, return_inverse=True); NB = len(ub)
def block_stats(pred, prev=LULC['2013'], true=LULC['2023']):
    t, p, q = true[BM], pred[BM], prev[BM]; ch = t != q
    comp = [np.ones_like(t), t == p, ch & (p == t), ch & (p == q), ch & (p != q) & (p != t), (~ch) & (p != q)]
    return np.stack([np.bincount(inv, weights=c.astype(float), minlength=NB) for c in comp], 1)
ST = {('-', 'Persistence', 'noshield'): block_stats(LULC['2013'])}
for (mode, tag), maps in HIND.items():
    e = vote(maps); ST[(mode, tag, 'noshield')] = block_stats(e); ST[(mode, tag, 'shield')] = block_stats(shield(e, 'hind'))
def oa_fom(s): return s[1] / s[0], s[2] / max(1, s[2:].sum())
def contrast(a, b, n=2000, seed=0):
    if a not in ST or b not in ST: return None
    rng = np.random.RandomState(seed); d = []
    for _ in range(n):
        s = rng.randint(0, NB, NB); oa1, f1 = oa_fom(ST[a][s].sum(0)); oa2, f2 = oa_fom(ST[b][s].sum(0)); d.append([oa1 - oa2, f1 - f2])
    d = np.array(d); lo, hi = np.percentile(d, [2.5, 97.5], 0)
    return dict(A='/'.join(a), B='/'.join(b), dOA=d[:, 0].mean(), OA_lo=lo[0], OA_hi=hi[0], dFoM=d[:, 1].mean(), FoM_lo=lo[1], FoM_hi=hi[1])
P0 = ('-', 'Persistence', 'noshield'); cr = []
for mode in MODES:
    for var in ['noshield', 'shield']:
        for tag in TAGS: cr.append(contrast((mode, tag, var), P0))
        for a, b in [('CNN_Markov', 'CA_ANN_Markov'), ('CNN_argmax', 'CA_ANN_Markov'), ('CNN_Markov', 'CNN_argmax'), ('PI_CNN_argmax', 'CNN_argmax'), ('PI_CNN_Markov', 'CNN_Markov')]:
            cr.append(contrast((mode, a, var), (mode, b, var)))
for tag in TAGS:
    for var in ['noshield', 'shield']:
        for other in ['global', 'anomaly_noclim', 'anomaly_stateonly']: cr.append(contrast(('anomaly', tag, var), (other, tag, var)))   # ablations: anomaly vs global scaling / without climate / state+static only
cr = pd.DataFrame([c for c in cr if c]); cr['sig'] = ~((cr.FoM_lo <= 0) & (cr.FoM_hi >= 0)) | ~((cr.OA_lo <= 0) & (cr.OA_hi >= 0))
print(f'\nBlock-bootstrap ({BOOT_REGION}, {NB} blocks) differences A - B (95% CI). sig = CI excludes 0 for OA or FoM:'); display(cr.round(5)); cr.to_csv(os.path.join(REP, 'bootstrap_contrasts.csv'), index=False)

# ---------- 3. FORECAST 2033 ----------
A23 = {n: float((LULC['2023'] == k).sum() * PIXEL_KM2) for k, n in enumerate(CLASS_NAMES)}
frows, FORE = [], {}
for mode in MODES:
    for tag, scs in TAGS.items():
        for sc in scs:
            mp = [read_idx(folder(mode), f'{tag}_fore2033_{sc}_seed{s}.tif') for s in SEEDS if ok(mode, f'{tag}_fore2033_{sc}_seed{s}.tif')]
            if not mp: continue
            FORE[(mode, tag, sc)] = mp
            for s, m in zip(SEEDS, mp):
                for var, mm in [('noshield', m), ('shield', shield(m, 'fore'))]:
                    d = dict(mode=mode, model=tag, scenario=sc, variant=var, seed=s, pct_changed=(mm[VALID] != LULC['2023'][VALID]).mean() * 100,
                             F2F_km2=float(((LULC['2023'] == FALLOW_IDX) & (mm == FOREST_IDX) & VALID).sum() * PIXEL_KM2),
                             new_water_steep_px=int(((mm == WATER_IDX) & (LULC['2023'] != WATER_IDX) & (slope_deg > STEEP_DEG) & VALID).sum()))
                    for k, n in enumerate(CLASS_NAMES): d[f'd_{n}'] = float((mm == k).sum() * PIXEL_KM2) - A23[n]
                    frows.append(d)
fc = pd.DataFrame(frows); fc.to_csv(os.path.join(REP, 'forecast_change_all.csv'), index=False)
gcols = ['pct_changed', 'F2F_km2', 'new_water_steep_px'] + [f'd_{n}' for n in CLASS_NAMES]
gg = fc.groupby(['mode', 'model', 'scenario', 'variant'])[gcols]
fsum = gg.mean().round(2).astype(str) + ' ± ' + gg.std().fillna(0).round(2).astype(str)
print(f'\nFORECAST 2023->2033: mean ± SD over seeds of change in km2 (observed 2013->2023 change = '
      f'{(LULC["2013"][VALID] != LULC["2023"][VALID]).mean()*100:.2f}% of pixels; observed Fallow->Forest 2013-23 = '
      f'{((LULC["2013"]==FALLOW_IDX)&(LULC["2023"]==FOREST_IDX)&VALID).sum()*PIXEL_KM2:.1f} km2)'); display(fsum); fsum.to_csv(os.path.join(REP, 'forecast_summary_mean_sd.csv'))

# ---------- 4. Trend figure with seed band (report mode, primary scenario, no shield) ----------
def trend_fig(tag, sc, mode=REPORT_MODE, fname=None):
    if (mode, tag, sc) not in FORE: print('no forecast for', mode, tag, sc); return
    per = pd.DataFrame([{n: float(((m == k) & VALID).sum() * PIXEL_KM2) for k, n in enumerate(CLASS_NAMES)} for m in FORE[(mode, tag, sc)]])
    obs = pd.DataFrame({y: {n: float(((LULC[y] == k) & VALID).sum() * PIXEL_KM2) for k, n in enumerate(CLASS_NAMES)} for y in YEARS}).T
    plt.rcParams['font.family'] = 'serif'; fig, ax = plt.subplots(2, 3, figsize=(15, 8)); ax = ax.ravel()
    for i, n in enumerate(CLASS_NAMES):
        ax[i].plot([0, 1, 2], obs[n].values, 'o-', lw=2.5, color='k', label='Observed')
        ax[i].plot([2, 3], [obs[n].values[2], per[n].mean()], 'o--', lw=2.5, color='C3', label=f'Forecast ({tag}, {sc})')
        ax[i].errorbar([3], [per[n].mean()], yerr=[[per[n].mean() - per[n].min()], [per[n].max() - per[n].mean()]], color='C3', capsize=5)
        ax[i].set_xticks([0, 1, 2, 3]); ax[i].set_xticklabels(['2003', '2013', '2023', '2033']); ax[i].set_title(n, fontweight='bold'); ax[i].grid(alpha=.3, ls='--'); ax[i].set_ylabel('km$^2$')
    ax[0].legend(fontsize=9); fig.delaxes(ax[5]); plt.tight_layout()
    if fname: plt.savefig(os.path.join(REP, fname), dpi=300, bbox_inches='tight')
    plt.show()
for tag in TAGS: trend_fig(tag, PRIMARY_SCEN if TAGS[tag][0] != 'na' else 'na', fname=f'trend_{tag}.png')
print('\nAll report tables/figures ->', REP)
